# Attention — The Library Version

Three parts, honestly framed: (1) verify the vectorized attention against a naive position-by-position loop — same numbers or bust; (2) a strong classical baseline on the key-value task, to show content lookup is genuinely the new capability; (3) the PyTorch translation, shown not run.

In [1]:
import numpy as np
import pandas as pd

def softmax(Z):
    Z = Z - Z.max(axis=-1, keepdims=True)
    e = np.exp(Z)
    return e / e.sum(axis=-1, keepdims=True)

# (1) vectorized vs naive-loop attention — the internal consistency check
rs = np.random.default_rng(0)
m, T, d = 8, 12, 16
X = rs.normal(0, 1, (m, T, d))
Wq, Wk, Wv = (rs.normal(0, 0.3, (d, d)) for _ in range(3))
Q, K, V = X @ Wq, X @ Wk, X @ Wv
O_vec = softmax(Q @ K.transpose(0, 2, 1) / np.sqrt(d)) @ V

O_loop = np.zeros_like(O_vec)
for i in range(m):
    for t in range(T):                                  # one query at a time, the slow honest way
        scores = np.array([Q[i, t] @ K[i, u] / np.sqrt(d) for u in range(T)])
        w = np.exp(scores - scores.max()); w = w / w.sum()
        O_loop[i, t] = sum(w[u] * V[i, u] for u in range(T))
print(f"vectorized vs naive loop — max |difference|: {np.abs(O_vec - O_loop).max():.2e}")
print("(identical: the matrix form IS the per-query lookup, batched)")

vectorized vs naive loop — max |difference|: 1.78e-15
(identical: the matrix form IS the per-query lookup, batched)


In [2]:
# (2) can a strong classical model do content-based lookup? (README §2.5)
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

kv = pd.read_csv("data/kv_data.csv")
LETTERS = list("ABCDEF")
PAIRS = {f"{L}{v}": i for i, (L, v) in enumerate((L, v) for L in LETTERS for v in range(10))}
QTOK = {L: 60 + i for i, L in enumerate(LETTERS)}
tok = np.array([[PAIRS[s[i:i+2]] for i in range(0, 8, 2)] + [QTOK[s[8]]] for s in kv.sequence])
Xoh = np.eye(66)[tok].reshape(len(tok), -1)             # one-hot, flattened: 330 features
Xtr, Xte, ytr, yte = train_test_split(Xoh, kv.answer.values, test_size=0.2, random_state=113)
mlp = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=400, random_state=0).fit(Xtr, ytr)
acc = accuracy_score(yte, mlp.predict(Xte))
print(f"sklearn MLP (128,64) on flattened key-value task: {acc:.0%}   (chance = 10%)")
print("Our single-head attention: 100%. The MLP must memorize letter-x-position-x-digit")
print("combinations case by case; attention COMPUTES the lookup — one mechanism, any position.")
n_mlp = sum(w.size for w in mlp.coefs_) + sum(b.size for b in mlp.intercepts_)
print(f"And the ledger: the MLP holds {n_mlp:,} parameters vs our attention model's ~4,800 —")
print(f"~{n_mlp//4800}x more, for {acc:.0%} vs 100%. Memorizing combinations is expensive;")
print("computing the lookup is cheap (README §2.5).")

sklearn MLP (128,64) on flattened key-value task: 52%   (chance = 10%)
Our single-head attention: 100%. The MLP must memorize letter-x-position-x-digit
combinations case by case; attention COMPUTES the lookup — one mechanism, any position.
And the ledger: the MLP holds 51,274 parameters vs our attention model's ~4,800 —
~10x more, for 52% vs 100%. Memorizing combinations is expensive;
computing the lookup is cheap (README §2.5).


### (3) The PyTorch translation — read it; you have built every line

```python
import torch, torch.nn as nn

attn = nn.MultiheadAttention(embed_dim=24, num_heads=1, batch_first=True)
out, weights = attn(X, X, X)        # self-attention: Q=K=V source — Blocks 4+6, autograd included
# 'weights' is our A — the budget matrix you read strategies from (Blocks 8–9)

# or the fused kernel behind modern LLMs (no weights returned — speed over legibility):
out = nn.functional.scaled_dot_product_attention(Q, K, V)
```

`num_heads=1` is this lesson exactly; `num_heads=8` is lesson 14's first move. The formula inside the fused kernel is Block 4's six lines — flash-attention and friends are memory-access engineering around the same math.